# NazarAI: train & evaluate the DR grading model (Kaggle GPU)

Runs the full pipeline from `ml/src/nazarai_ml` (the same preprocessing module the deployed service uses):
grade-stratified train/validation split of **APTOS 2019** → EfficientNet-B3 ordinal regression →
threshold optimisation on **validation only** → ONNX export (parity-checked) →
evaluation on validation + **external IDRiD test** → `metrics.json` + plots.

**Data plan (South Asian data, closest public match to Pakistani patients):**
- Train + validation: **APTOS 2019** (Aravind Eye Hospital, India; 3,662 graded images, many cameras).
- External test: **IDRiD** (Nanded, India; 516 graded images), never used for training or threshold tuning.

**Notebook settings:** Accelerator = GPU T4, Internet = On.

**Attach these inputs** (Add Input):
1. Competition: `aptos2019-blindness-detection` (**required**; accept its rules on Kaggle first)
2. Dataset: `<you>/nazarai-ml-src` (the `nazarai_ml` package files, uploaded from this repo's `ml/src/nazarai_ml`)
3. *(optional, off by default)* Dataset: `tanlikesmath/diabetic-retinopathy-resized` (pre-resized EyePACS 2015), only if `USE_EYEPACS = True`

IDRiD (CC BY 4.0) is downloaded from its Hugging Face mirror and used **only** as the external test set.

Budget: APTOS-only at 384 px is roughly 2–3 min per epoch on a T4 (full-resolution PNG decoding dominates).

In [ ]:
# ---- configuration ---------------------------------------------------------
REPO_URL = "https://github.com/InitialCommit-CYE/nazarai.git"   # only used if the nazarai-ml-src dataset is not attached
BACKBONE = "efficientnet_b3"
IMAGE_SIZE = 384
EPOCHS = 15              # APTOS is small (~3.1k training images), so more passes than the EyePACS run
BATCH_SIZE = 24          # T4 16 GB at 384 px with mixed precision
USE_EYEPACS = False      # training data = APTOS 2019 only (team decision 2026-11-25)
MAX_EYEPACS = 0          # only used when USE_EYEPACS: 0 = all ~35k images
TARGET_SENSITIVITY = 0.90  # referral threshold chosen on validation (IDx-DR reported 0.87)
WORK = "/kaggle/working"

In [ ]:
import glob, os, shutil, subprocess, sys
!pip install -q onnx onnxruntime grad-cam huggingface_hub
# The nazarai_ml package: from the attached dataset (files may sit at any depth), else git clone.
hits = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/preprocess.py", recursive=True)
        if os.path.exists(os.path.join(os.path.dirname(p), "quality.py"))]
SRC = f"{WORK}/src"
if hits:
    shutil.copytree(hits[0], f"{SRC}/nazarai_ml", dirs_exist_ok=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, f"{WORK}/nazarai"], check=True)
    SRC = f"{WORK}/nazarai/ml/src"
sys.path.insert(0, SRC)
os.environ["PYTHONPATH"] = SRC
os.chdir(WORK)
import nazarai_ml, torch
print("nazarai_ml from", SRC, "| torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

In [ ]:
# ---- locate datasets -------------------------------------------------------
def find(pattern):
    hits = sorted(glob.glob(pattern, recursive=True))
    return hits[0] if hits else None

aptos_csv = find("/kaggle/input/**/aptos2019*/train.csv")
APTOS = os.path.dirname(aptos_csv) if aptos_csv else None
print("APTOS:", APTOS)
assert APTOS, "attach the aptos2019-blindness-detection competition (accept its rules first)"
EYEPACS_ARG = ""
if USE_EYEPACS:
    EYEPACS_CSV = find("/kaggle/input/**/trainLabels_cropped.csv") or find("/kaggle/input/**/trainLabels.csv")
    EYEPACS_IMG = (find("/kaggle/input/**/resized_train_cropped/resized_train_cropped")
                   or find("/kaggle/input/**/resized_train/resized_train"))
    assert EYEPACS_CSV and EYEPACS_IMG, "USE_EYEPACS=True but the resized EyePACS dataset is not attached"
    EYEPACS_ARG = f'--eyepacs-csv "{EYEPACS_CSV}" --eyepacs-images "{EYEPACS_IMG}" --max-eyepacs {MAX_EYEPACS}'
print("training sources:", "APTOS 2019 + EyePACS" if USE_EYEPACS else "APTOS 2019 only")

In [ ]:
# ---- external test set: IDRiD (all 516 graded images) ----------------------
!python -m nazarai_ml.idrid_hf --out {WORK}/idrid

In [ ]:
%%time
# ---- manifests: grade-stratified split (grouped by patient where IDs exist; APTOS has none) ----
!python -m nazarai_ml.dataset build --aptos-root "{APTOS}" {EYEPACS_ARG} --idrid-csv {WORK}/idrid/labels.csv --idrid-images {WORK}/idrid/images --out {WORK}/manifests

In [ ]:
# ---- train (best checkpoint by validation QWK; thresholds fitted on validation) ----
!python -m nazarai_ml.train --manifests {WORK}/manifests --out {WORK}/run --backbone {BACKBONE} --size {IMAGE_SIZE}     --epochs {EPOCHS} --batch-size {BATCH_SIZE} --workers 4 --target-sensitivity {TARGET_SENSITIVITY}

In [ ]:
# ---- export to ONNX (fails loudly if outputs differ from PyTorch) ----------
!python -m nazarai_ml.export_onnx --run {WORK}/run

In [ ]:
# ---- evaluate exactly as served: ONNX on CPU, 4-view TTA, deployed referral rule ----
!python -m nazarai_ml.evaluate --run {WORK}/run --manifests {WORK}/manifests --reports {WORK}/reports

In [ ]:
import json, shutil
from IPython.display import Image, display
m = json.load(open(f"{WORK}/reports/metrics.json"))
for k, s in m["splits"].items():
    r = s["referable"]
    print(f"{s['name']:28s} n={s['n_images']:5d}  QWK={s['qwk']:.3f}  sens={r['sensitivity']}  spec={r['specificity']}  AUC={r['auc']}")
print("IDx-DR reference: sens 0.87 / spec 0.90 | mean CPU latency:", m["inference"]["mean_ms"], "ms")
display(Image(f"{WORK}/reports/roc.png"))
os.makedirs(f"{WORK}/release", exist_ok=True)
for f in ["model.onnx", "model.pt", "model_config.json", "history.json"]:
    shutil.copy(f"{WORK}/run/{f}", f"{WORK}/release/{f}")
shutil.copytree(f"{WORK}/reports", f"{WORK}/release/reports", dirs_exist_ok=True)
print("Download /kaggle/working/release: commit metrics.json + PNGs to ml/reports/, upload weights with scripts/upload_model.py")

## Optional: publish the weights straight to the Hugging Face Hub
Add a Kaggle secret `HF_TOKEN` (write access), then run the cell below. The deployed ML service downloads
`model.onnx`, `model.pt` and `model_config.json` from this repo at startup (`HF_MODEL_REPO`).

In [ ]:
HF_MODEL_REPO = "<your-hf-user>/nazarai-dr-model"   # set this, and add a Kaggle secret HF_TOKEN (write access)
if "<" in HF_MODEL_REPO:
    print("HF upload skipped: set HF_MODEL_REPO to publish the weights (or use scripts/upload_model.py locally)")
else:
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
        from huggingface_hub import HfApi
        api = HfApi(); api.create_repo(HF_MODEL_REPO, private=True, exist_ok=True)
        for f in ["model.onnx", "model.pt", "model_config.json"]:
            api.upload_file(path_or_fileobj=f"{WORK}/release/{f}", path_in_repo=f, repo_id=HF_MODEL_REPO)
        api.upload_file(path_or_fileobj=f"{WORK}/release/reports/metrics.json", path_in_repo="metrics.json", repo_id=HF_MODEL_REPO)
        print("uploaded to", HF_MODEL_REPO)
    except Exception as e:
        print("HF upload skipped:", type(e).__name__, e)